# VWAP and five-condition backtest

Compares 2–5 confirmations using previous close, daily volatility, VWAP and moving averages.

**Status: exploratory research; not production validated.**

## Limitations
Default transaction costs and slippage are zero. Position sizing uses fractional 1x notional exposure; it does not model actual WIN/WDO contract multipliers, margin or integer contracts. There is no fixed stop/target. Session completeness filters can exclude observations. Results are exploratory, without an untouched out-of-sample evaluation.

Original source: `backtest5variaveis.ipynb`. Outputs cleared for publication. Analytical code preserved.


In [ ]:
"""Indicador sem pesos + grade + backtest intraday.
Instalacao:
  pip install pandas numpy plotly
  pip install --upgrade git+https://github.com/rongardF/tvdatafeed.git
Jupyter:
  from indicador_vwap import executar, Config
  r = executar(Config())
  display(r['grade']); display(r['anual']); display(r['mensal'])
Terminal: python indicador_vwap.py

Convencoes:
- Direcao obrigatoria e pelo menos 2, 3, 4 ou 5 confirmacoes, sem pesos.
- Vol prevista = desvio-padrao de 20 retornos diarios simples, ate D-1.
- Distancia do fechamento D-1 limitada a 0,5 sigma diario (nao amplitude).
- SMA20 > SMA50 e ambas subindo em relacao a 3 candles atras; inverso venda.
- VWAP por HLC3 e volume. Toque aproximado por low <= VWAP <= high;
  ignora somente o primeiro candle do dia. Depois de um toque bloqueia o dia.
- Sinal no fechamento, entrada/saida na abertura seguinte; neutro nao zera.
- Zera no fechamento do ultimo candle da sessao configurada, sem stop/alvo.
- Quantidade fixa por trade: nocional = patrimonio na entrada (1x, fracionario).
  Retorno percentual nocional, nao retorno sobre margem/contratos WIN reais.
- So testa dias completos com todos os candles esperados. Sem overnight.
- Custos percentuais por lado e slippage em pontos configuraveis.
- Grade de candles fechados, snapshot; nao e streaming nem indicador NTSL.
- Serie continua pode ter distorcoes de rolagem. Volume obrigatorio.
- Horario configurado deve corresponder a sessao historica do ativo.
"""
from dataclasses import dataclass
from pathlib import Path
import os
import time
import numpy as np
import pandas as pd

@dataclass
class Config:
    ativos: tuple = ('WIN1!',)
    bolsa: str = 'BMFBOVESPA'
    minutos: int = 15
    barras: int = 5000
    dias_vol: int = 20
    fracao_vol: float = .5
    inclinacao: int = 3
    inicio: str = '09:00'
    fim: str = '18:00'
    timezone: str = 'America/Sao_Paulo'
    # tvDatafeed usa datetime.fromtimestamp; fusos naive dependem da maquina.
    # Se None, usa timezone local do computador para converter os timestamps.
    timezone_fonte: str = None
    custo_lado: float = 0.0
    slippage_pontos: float = 0.0
    capital: float = 10000.0
    pasta: str = 'resultado_vwap'
    mostrar_graficos: bool = True


def normalizar(df, cfg):
    if df is None or df.empty:
        raise ValueError('Fonte retornou dados vazios. Confira simbolo, bolsa e acesso.')
    x = df.copy().sort_index()
    idx = pd.DatetimeIndex(x.index)
    if idx.tz is None:
        if cfg.timezone_fonte:
            idx = idx.tz_localize(cfg.timezone_fonte, ambiguous='raise', nonexistent='raise')
        else:
            # Reverte o datetime local produzido por datetime.fromtimestamp.
            idx = pd.to_datetime([v.to_pydatetime().timestamp() for v in idx], unit='s', utc=True)
    x.index = idx.tz_convert(cfg.timezone)
    if x.index.has_duplicates:
        raise ValueError('Timestamps duplicados na fonte.')
    cols = ['open', 'high', 'low', 'close', 'volume']
    x = x[cols].apply(pd.to_numeric, errors='coerce')
    if not np.isfinite(x.to_numpy()).all() or (x[cols[:4]] <= 0).any().any() or (x.volume < 0).any():
        raise ValueError('OHLCV invalido ou ausente.')
    if ((x.high < x[['open','close','low']].max(axis=1)) | (x.low > x[['open','close','high']].min(axis=1))).any():
        raise ValueError('OHLC inconsistente.')
    return x


def baixar(ativo, cfg):
    from tvDatafeed import TvDatafeed, Interval
    intervalos = {5: Interval.in_5_minute, 15: Interval.in_15_minute,
                  30: Interval.in_30_minute, 60: Interval.in_1_hour}
    tv = TvDatafeed(os.environ.get('TV_USER'), os.environ.get('TV_PASSWORD'))
    intraday = tv.get_hist(symbol=ativo, exchange=cfg.bolsa,
                          interval=intervalos[cfg.minutos], n_bars=cfg.barras)
    daily = tv.get_hist(symbol=ativo, exchange=cfg.bolsa,
                       interval=Interval.in_daily, n_bars=1500)
    return normalizar(intraday, cfg), normalizar(daily, cfg)


def indicadores(intra, diario, cfg, agora=None):
    agora = pd.Timestamp.now(tz=cfg.timezone) if agora is None else pd.Timestamp(agora)
    x = intra.loc[intra.index + pd.Timedelta(minutes=cfg.minutos) <= agora].copy()
    x = x.between_time(cfg.inicio, cfg.fim, inclusive='left')
    if x.empty:
        raise ValueError('Sem candles fechados no horario configurado.')
    x['dia'] = x.index.date
    d = diario.copy()
    d['dia'] = d.index.date
    d = d.groupby('dia').last()
    # Shift antes do mapeamento: nao usa fechamento nem volatilidade do proprio dia.
    prev = d.close.shift(1)
    sigma = d.close.pct_change(fill_method=None).rolling(cfg.dias_vol).std(ddof=1).shift(1)
    x['fech_anterior'] = x.dia.map(prev)
    x['vol_dia'] = x.dia.map(sigma)
    x['distancia'] = x.close / x.fech_anterior - 1
    x['limite'] = cfg.fracao_vol * x.vol_dia
    tpv = (x.high + x.low + x.close) / 3 * x.volume
    cv = x.volume.groupby(x.dia).cumsum()
    x['vwap'] = tpv.groupby(x.dia).cumsum() / cv.replace(0, np.nan)
    ordinal = x.groupby('dia').cumcount()
    tocou = (ordinal > 0) & (x.low <= x.vwap) & (x.high >= x.vwap)
    x['sem_toque'] = ~tocou.groupby(x.dia).cummax()
    x['mm20'] = x.close.rolling(20).mean()
    x['mm50'] = x.close.rolling(50).mean()
    x['alta'] = (x.mm20 > x.mm50) & (x.mm20 > x.mm20.shift(cfg.inclinacao)) & (x.mm50 > x.mm50.shift(cfg.inclinacao))
    x['baixa'] = (x.mm20 < x.mm50) & (x.mm20 < x.mm20.shift(cfg.inclinacao)) & (x.mm50 < x.mm50.shift(cfg.inclinacao))
    # Primeiro dia truncado nao pode comprovar ausencia de toque anterior.
    primeira = x.groupby('dia').apply(lambda g: g.index[0].strftime('%H:%M'))
    x['historico_sessao'] = x.dia.map(primeira.eq(cfg.inicio))
    gaps = x.index.to_series().diff().gt(pd.Timedelta(minutes=cfg.minutos)) & ordinal.gt(0)
    x['historico_sessao'] &= ~gaps.groupby(x.dia).cummax()
    valido = x[['fech_anterior','vol_dia','vwap','mm50']].notna().all(axis=1) & (x.vol_dia > 0) & x.historico_sessao & (cv > 0)
    x['valido'] = valido & (ordinal > 0)
    x['direcao_compra'] = x.distancia > 0
    x['direcao_venda'] = x.distancia < 0
    x['espaco'] = x.distancia.abs() <= x.limite
    x['vwap_compra'] = x.close >= x.vwap
    x['vwap_venda'] = x.close <= x.vwap
    x['confirmacoes_compra'] = x[['direcao_compra','espaco','sem_toque','vwap_compra','alta']].astype(int).sum(axis=1)
    x['confirmacoes_venda'] = x[['direcao_venda','espaco','sem_toque','vwap_venda','baixa']].astype(int).sum(axis=1)
    x['estado'] = 'NEUTRO'
    for lado, direcao in [('COMPRA','compra'),('VENDA','venda')]:
        mask = x.valido & x['direcao_'+direcao] & (x['confirmacoes_'+direcao]>=2)
        x.loc[mask,'estado'] = lado+' '+x.loc[mask,'confirmacoes_'+direcao].astype(str)+'/5'
    x.loc[~x.valido,'estado'] = 'SEM DADOS'
    return x


def simular(x, cfg, nivel=5, somente=0):
    x = x.copy()
    cp = x.valido & x.direcao_compra & (x.confirmacoes_compra >= nivel)
    vd = x.valido & x.direcao_venda & (x.confirmacoes_venda >= nivel)
    x['sinal'] = np.select([cp,vd],[1,-1],default=0)
    equity = cfg.capital
    registros, curva, ignorados = [], [], []
    for dia, g in x.groupby('dia'):
        start = pd.Timestamp(f'{dia} {cfg.inicio}', tz=cfg.timezone)
        end = pd.Timestamp(f'{dia} {cfg.fim}', tz=cfg.timezone)
        esperado = pd.date_range(start, end, freq=f'{cfg.minutos}min', inclusive='left')
        if not g.index.equals(esperado) or g[['vol_dia','mm50']].isna().any().any():
            ignorados.append(str(dia)); continue
        lado = 0
        for j, (t, row) in enumerate(g.iterrows()):
            desejado = int(g.sinal.iloc[j-1]) if j > 0 else 0
            # Nao abre no ultimo candle; permite encerrar posicao existente.
            fechar = lado != 0 and desejado == -lado
            if fechar:
                px = row.open - lado * cfg.slippage_pontos
                pnl = qtd * lado * (px - ent) - qtd * px * cfg.custo_lado
                equity += pnl
                registros.append([entrada,t,lado,ent,px,equity-base,(equity/base-1),'contrario'])
                lado = 0
            if lado == 0 and desejado and (somente == 0 or desejado == somente) and j < len(g)-1:
                lado, entrada, base = desejado, t, equity
                ent = row.open + lado * cfg.slippage_pontos
                qtd = base / ent
                equity -= qtd * ent * cfg.custo_lado
            if lado and j == len(g)-1:
                px = row.close - lado * cfg.slippage_pontos
                equity += qtd * lado * (px-ent) - qtd * px * cfg.custo_lado
                registros.append([entrada,t+pd.Timedelta(minutes=cfg.minutos),lado,ent,px,equity-base,equity/base-1,'fim_dia'])
                lado = 0
            mtm = equity + (qtd * lado * (row.close-ent) if lado else 0)
            if mtm <= 0:
                raise ValueError('Capital esgotado.')
            curva.append([t+pd.Timedelta(minutes=cfg.minutos),mtm])
    tr = pd.DataFrame(registros, columns=['entrada','saida','lado','preco_entrada','preco_saida','pnl','retorno','motivo'])
    eq = pd.DataFrame(curva, columns=['data','capital']).set_index('data')
    if eq.empty:
        raise ValueError('Nenhuma sessao completa utilizavel; confira horario, timeframe e historico.')
    eq['retorno'] = eq.capital.pct_change()
    eq.iloc[0,eq.columns.get_loc('retorno')] = eq.capital.iloc[0]/cfg.capital-1
    eq['drawdown'] = eq.capital/eq.capital.cummax().clip(lower=cfg.capital)-1
    return tr, eq, ignorados


def tabela(tr, eq, freq):
    keys = eq.index.strftime('%Y-%m' if freq=='mes' else '%Y')
    r = eq.groupby(keys).agg(inicio=('capital',lambda s:s.index.min()),fim=('capital',lambda s:s.index.max()))
    r['retorno_pct'] = ((1+eq.retorno).groupby(keys).prod()-1)*100
    r['trades'] = 0
    r['acerto_pct'] = np.nan
    if not tr.empty:
        k = tr.saida.dt.strftime('%Y-%m' if freq=='mes' else '%Y')
        r['trades'] = tr.groupby(k).size().reindex(r.index,fill_value=0)
        r['acerto_pct'] = tr.retorno.gt(0).groupby(k).mean().reindex(r.index)*100
    r['cobertura'] = 'periodo observado; confira inicio/fim e sessoes excluidas'
    return r


def metricas(tr, eq, cfg):
    ganhos = tr.loc[tr.pnl>0,'pnl'].sum() if not tr.empty else 0
    perdas = -tr.loc[tr.pnl<0,'pnl'].sum() if not tr.empty else 0
    return {'retorno_pct':(eq.capital.iloc[-1]/cfg.capital-1)*100,
            'trades':len(tr), 'acerto_pct':(tr.retorno.gt(0).mean()*100 if len(tr) else np.nan),
            'media_trade_pct':(tr.retorno.mean()*100 if len(tr) else np.nan),
            'profit_factor':(ganhos/perdas if perdas else np.inf if ganhos else np.nan),
            'drawdown_pct':eq.drawdown.min()*100}


def grafico_preco(x, tr, ativo, nivel):
    import plotly.graph_objects as go
    fig = go.Figure(go.Candlestick(x=x.index,open=x.open,high=x.high,low=x.low,close=x.close,name='Preco'))
    for col, nome, cor in [('vwap','VWAP','#c08000'),('mm20','Media 20','#1976d2'),('mm50','Media 50','#ab47bc')]:
        fig.add_trace(go.Scatter(x=x.index,y=x[col],name=nome,line=dict(color=cor,width=1)))
    for lado,nome,cor,simbolo in [(1,'Entrada compra','#00a050','triangle-up'),(-1,'Entrada venda','#d32f2f','triangle-down')]:
        t=tr[tr.lado==lado]
        fig.add_trace(go.Scatter(x=t.entrada,y=t.preco_entrada,mode='markers',name=nome,
                                marker=dict(color=cor,size=12,symbol=simbolo)))
    fig.add_trace(go.Scatter(x=tr.saida,y=tr.preco_saida,mode='markers',name='Saida',
                            text=tr.motivo,marker=dict(color='#222222',size=9,symbol='x')))
    fig.update_layout(title=f'{ativo} — pelo menos {nivel}/5 — operacoes executadas',
                      template='plotly_white',height=700,xaxis_rangeslider_visible=False,
                      xaxis_title='Horario da bolsa',yaxis_title='Preco')
    return fig


def executar(cfg=None):
    import plotly.graph_objects as go
    cfg = cfg or Config()
    if cfg.minutos not in (5,15,30,60) or not 2 <= cfg.barras <= 5000:
        raise ValueError('Use 5/15/30/60 minutos e 2 a 5000 barras.')
    if cfg.capital <= 0 or not 0 <= cfg.custo_lado < 1 or cfg.slippage_pontos < 0:
        raise ValueError('Capital/custos invalidos.')
    pasta=Path(cfg.pasta); pasta.mkdir(parents=True,exist_ok=True)
    grades, resumos, anuais, mensais, combinacoes, detalhes, erros = [],[],[],[],[],{},[]
    for ativo in cfg.ativos:
        try:
            intra,diario=baixar(ativo,cfg)
            x=indicadores(intra,diario,cfg)
            last=x.iloc[-1]
            grades.append({'ativo':ativo,'candle':x.index[-1],'sinal':last.estado,
                           'preco':last.close,'confirmacoes_compra':last.confirmacoes_compra,
                           'confirmacoes_venda':last.confirmacoes_venda,
                           'direcao_compra':last.direcao_compra,'direcao_venda':last.direcao_venda,
                           'espaco':last.espaco,'sem_toque':last.sem_toque,
                           'vwap_compra':last.vwap_compra,'vwap_venda':last.vwap_venda,
                           'alta':last.alta,'baixa':last.baixa,
                           'distancia_pct':last.distancia*100,'limite_pct':last.limite*100})
            safe=''.join(c for c in ativo if c.isalnum())
            fig_cap=go.Figure()
            detalhes[ativo]={}
            for nivel in (2,3,4,5):
                # Simulacoes independentes; resultados por lado nao sao somaveis.
                for filtro,nome in [(0,'Conjunto'),(1,'Compras'),(-1,'Vendas')]:
                    tr,eq,ignorados=simular(x,cfg,nivel,filtro)
                    meta={'ativo':ativo,'nivel':nivel,'operacional':nome}
                    resumos.append(dict(meta,**metricas(tr,eq,cfg)))
                    for freq,lista in [('ano',anuais),('mes',mensais)]:
                        tab=tabela(tr,eq,freq)
                        for key,value in meta.items(): tab[key]=value
                        lista.append(tab)
                    prefix=f'{safe}_{nivel}_{nome}'
                    tr.to_csv(pasta/f'{prefix}_trades.csv',index=False)
                    eq.to_csv(pasta/f'{prefix}_curva.csv')
                    if filtro==0:
                        # Condicoes do candle do sinal, nao da abertura executada.
                        if not tr.empty:
                            entradas=x.index.get_indexer(pd.DatetimeIndex(tr.entrada))-1
                            masks=[]
                            for k,(_,oper) in zip(entradas,tr.iterrows()):
                                cols=['espaco','sem_toque','vwap_compra','alta'] if oper.lado==1 else ['espaco','sem_toque','vwap_venda','baixa']
                                masks.append('direcao+'+'+'.join(c for c in cols if bool(x.iloc[k][c])))
                            tr['combinacao']=masks
                            for (lado,comb),grupo in tr.groupby(['lado','combinacao']):
                                combinacoes.append(dict(meta,lado=lado,combinacao=comb,trades=len(grupo),
                                    acerto_pct=grupo.retorno.gt(0).mean()*100,media_trade_pct=grupo.retorno.mean()*100))
                            tr.to_csv(pasta/f'{prefix}_trades.csv',index=False)
                        fig=grafico_preco(x,tr,ativo,nivel)
                        fig.write_html(pasta/f'{safe}_entradas_{nivel}.html',include_plotlyjs=True)
                        if cfg.mostrar_graficos: fig.show()
                        fig_cap.add_trace(go.Scatter(x=eq.index,y=(eq.capital/cfg.capital-1)*100,name=f'{nivel}/5'))
                        detalhes[ativo][nivel]={'trades':tr,'curva':eq,'grafico':fig,'dias_excluidos':ignorados}
                print(f'{ativo}: nivel {nivel}/5 concluido.')
            fig_cap.update_layout(title=f'{ativo} — retorno acumulado por nivel',template='plotly_white',yaxis_title='Retorno (%)')
            fig_cap.write_html(pasta/f'{safe}_capital.html',include_plotlyjs=True)
            if cfg.mostrar_graficos: fig_cap.show()
            x.to_csv(pasta/f'{safe}_indicadores.csv')
            (pasta/f'{safe}_dias_excluidos.txt').write_text('\n'.join(ignorados))
        except Exception as exc:
            erros.append({'ativo':ativo,'erro':f'{type(exc).__name__}: {exc}'})
            print(f'ERRO {ativo}: {exc}')
    out={'grade':pd.DataFrame(grades),'comparativo':pd.DataFrame(resumos),
         'anual':pd.concat(anuais) if anuais else pd.DataFrame(),
         'mensal':pd.concat(mensais) if mensais else pd.DataFrame(),
         'combinacoes':pd.DataFrame(combinacoes),'erros':pd.DataFrame(erros)}
    for nome,tab in out.items():
        tab.to_csv(pasta/f'{nome}.csv',index=True)
        print('\n'+nome.upper()+'\n'+tab.to_string(float_format=lambda v:f'{v:.2f}'))
    colors={'COMPRA':'#c6efce','VENDA':'#ffc7ce','NEUTRO':'#eeeeee'}
    grade=out['grade']
    from html import escape
    html='<table border="1"><tr>'+''.join('<th>'+escape(str(c))+'</th>' for c in grade.columns)+'</tr>'
    for _,row in grade.iterrows():
        cor=colors.get(str(row.get('sinal')).split()[0],'#fff2cc')
        html+='<tr style="background-color:'+cor+'">'+''.join('<td>'+escape(str(v))+'</td>' for v in row)+'</tr>'
    html+='</table>'

    aviso='<p>Snapshot de candles fechados. Periodos podem ser parciais. Compras e vendas sao simulacoes independentes. Nocional 1x; nao representa margem de contratos WIN.</p>'
    tabelas=''.join('<h2>'+nome+'</h2>'+tab.to_html() for nome,tab in out.items() if nome!='grade')
    (pasta/'relatorio.html').write_text('<meta charset="utf-8"><h1>Comparacao 2 a 5 confirmacoes</h1>'+aviso+html+tabelas,encoding='utf-8')
    out['detalhes']=detalhes
    return out


if __name__ == '__main__':
    import sys
    if 'ipykernel' in sys.modules:
        print('Rode: resultados = executar(Config())')
    else:
        executar()


In [ ]:
resultados = executar(Config(
    ativos=("WDO1!",),
    minutos=5,
    mostrar_graficos=True,
))

display(resultados["comparativo"])
display(resultados["anual"])
display(resultados["mensal"])